# Ocean Wind 1

In [1]:
from pathlib import Path

from whale import Project
from ORBIT import ProjectManager
from ORBIT.core import library
from ORBIT.phases.design import CustomArraySystemDesign

In [2]:
# Define where the project data lives
library_path = Path("library").resolve()

# Define the configuration file with each model's loaded data
# config_file_name = "ocean_wind_example.yaml"
config_file_name = "Ocean_Wind_1_base_install"

library.initialize_library(str(library_path))
config = library.extract_library_specs("config", config_file_name)

ORBIT library intialized at '/Users/rhammond/Documents/GitHub/WHaLE/examples/library'


In [3]:
array = CustomArraySystemDesign(config)
array.run()

Missing data in columns ['cable_length', 'bury_speed']; all values will be calculated.

IndexError: index 22 is out of bounds for axis 0 with size 22

In [4]:
array.location_data

,id,substation_id,substation_name,substation_latitude,substation_longitude,turbine_name,turbine_latitude,turbine_longitude,string,order,cable_length,bury_speed
0,L107_A02,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A02,39.208283,-74.193189,0,0,0,0
1,L107_A03,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A03,39.198651,-74.181247,0,1,0,0
2,L107_A04,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A04,39.189017,-74.169309,0,2,0,0
3,L107_A05,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A05,39.179383,-74.157374,0,3,0,0
4,L107_A06,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A06,39.169748,-74.145442,0,4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
93,L107_I02,Substation Z11,Substation Z11,39.120386,-74.301084,L107_I02,39.119871,-74.328511,7,0,0,0
94,L107_I01,Substation Z11,Substation Z11,39.120386,-74.301084,L107_I01,39.130091,-74.339540,7,1,0,0
95,L107_J01,Substation Z11,Substation Z11,39.120386,-74.301084,L107_J01,39.119351,-74.355937,7,2,0,0
96,L107_K01,Substation Z11,Substation Z11,39.120386,-74.301084,L107_K01,39.108608,-74.372330,7,3,0,0


In [8]:
import numpy as np

array.location_data_x = np.zeros(
    (array.num_strings, array.num_turbines_full_string + 1), dtype=float
)
array.location_data_y = np.zeros(
    (array.num_strings, array.num_turbines_full_string + 1), dtype=float
)
array.sections_cable_lengths = np.zeros(
    (array.num_strings, array.num_turbines_full_string), dtype=float
)
array.sections_bury_speeds = np.zeros(
    (array.num_strings, array.num_turbines_full_string), dtype=float
)

array.oss_x = []
array.oss_y = []
i = 0
for oss in array.location_data.substation_id.unique():
    print(oss, i)
    layout = array.location_data[
        array.location_data.substation_id == oss
    ]
    string_id = np.sort(layout.string.unique())
    print(string_id, end=" | ")
    string_id += i
    print(string_id)

    x = layout.substation_longitude.values[0]
    y = layout.substation_latitude.values[0]
    array.oss_x.append(x)
    array.oss_y.append(y)
    array.location_data_x[string_id, 0] = x
    array.location_data_y[string_id, 0] = y
    for string in string_id:
        print(string)
        data = layout[layout.string == string - i]
        order = data["order"].values
        array.location_data_x[
            string, order + 1
        ] = data.turbine_longitude.values[order]
        array.location_data_y[
            string, order + 1
        ] = data.turbine_latitude.values[order]
        array.sections_cable_lengths[
            string, order
        ] = data.cable_length.values[order]
        array.sections_bury_speeds[
            string, order
            
        ] = data.bury_speed.values[order]
    print(i, string)
    i = string + 1

no_turbines = array.location_data_x == 0
array.location_data_x[no_turbines] = None
array.location_data_y[no_turbines] = None

array.sections_cable_lengths[no_turbines[:, 1:]] = None
array.sections_bury_speeds[no_turbines[:, 1:]] = None
array._check_optional_input()

array.coordinates = np.dstack(
    (array.location_data_x, array.location_data_y)
)

# Create the distances between each subsequent turbine in a string
if array.distance:
    array.sections_distance = array._compute_euclidean_distance()
else:
    array.sections_distance = array._compute_haversine_distance()

Substation Z01 0
[0 1 2 3 4 5 6] | [0 1 2 3 4 5 6]
0
1
2
3
4
5
6
0 6
Substation Z02 7
[0 1 2 3 4 5 6] | [ 7  8  9 10 11 12 13]
7
8
9
10
11
12
13
7 13
Substation Z11 14
[0 1 2 3 4 5 6 7] | [14 15 16 17 18 19 20 21]
14
15
16
17
18
19
20
21
14 21


In [26]:
i = 0
locations = array.location_data.copy()
for oss in array.location_data.substation_id.unique():
    oss_ix = array.location_data.substation_id == oss
    oss_layout = array.location_data.loc[oss_ix]
    string_id = np.sort(oss_layout.string.unique())
    for string in string_id:
        string_ix = oss_ix & (array.location_data.string == string)
        cable_order = array.location_data.loc[string_ix, "order"].values
        locations.loc[string_ix, "cable_length"] = array.sections_cable_lengths[string + i, cable_order]
    i = string + 1

Substation Z01 0
0 [0 1 2 3 4]
1 [0 1 2 3]
2 [0 1 2 3 4]
3 [0 1 2 3]
4 [0 1 2 3]
5 [0 1 2 3 4]
6 [0 1 2 3 4]
Substation Z02 7
0 [0 1 2 3]
1 [0 1 2 3 4]
2 [0 1 2 3 4]
3 [0 1 2 3 4]
4 [0 1 1 2]
5 [0 1 1 2]
6 [0 1 2 3]
Substation Z11 7
0 [0 1 2 3 4]
1 [0 1 2 3 4]
2 [0 1 2 3]
3 [0 1 2]
4 [0 1 2 3]
5 [0 1 2 3 4]
6 [0 1 2 3]
7 [0 1 2 3 4]


In [27]:
locations.head(20)

,id,substation_id,substation_name,substation_latitude,substation_longitude,turbine_name,turbine_latitude,turbine_longitude,string,order,cable_length,bury_speed
0,L107_A02,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A02,39.208283,-74.193189,0,0,2.462920,0
1,L107_A03,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A03,39.198651,-74.181247,0,1,1.537953,0
2,L107_A04,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A04,39.189017,-74.169309,0,2,1.537984,0
3,L107_A05,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A05,39.179383,-74.157374,0,3,1.537951,0
4,L107_A06,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A06,39.169748,-74.145442,0,4,1.537977,0
5,L107_A07,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A07,39.160111,-74.133514,1,0,6.444450,0
6,L107_A08,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A08,39.150473,-74.121588,1,1,1.537953,0
7,L107_A09,Substation Z01,Substation Z01,39.187079,-74.198994,L107_A09,39.140834,-74.109666,1,2,1.537962,0
8,L107_B09,Substation Z01,Substation Z01,39.187079,-74.198994,L107_B09,39.126270,-74.132018,1,3,2.570523,0
9,L107_B04,Substation Z01,Substation Z01,39.187079,-74.198994,L107_B04,39.176947,-74.187824,2,0,1.534773,0


In [13]:
array._create_cable_section_lengths()

In [18]:
array.sections_cable_lengths

array([[2.46291983, 1.53795301, 1.53798447, 1.53795073, 1.53797697],
       [6.44444983, 1.53795282, 1.53796165, 2.57052297,        nan],
       [1.53477271, 1.53478533, 1.53479876, 1.53477166, 1.53478557],
       [4.88485752, 1.53472729, 1.53476487, 1.53473189,        nan],
       [9.75744268, 1.53474999, 2.15920786, 1.534767  ,        nan],
       [2.44923901, 1.53477653, 2.53299342, 1.53474755, 1.53474584],
       [1.53477601, 1.91521493, 1.53474687, 2.53474547, 1.53475782],
       [3.68674117, 1.53476518, 1.5347628 , 1.53475366,        nan],
       [6.00635016, 1.53478357, 1.53477405, 1.53474846, 1.53478752],
       [8.3579001 , 1.53477846, 1.53476967, 1.53478249, 1.90395645],
       [2.62026756, 1.90397625, 1.53477072, 1.90395248, 1.53476586],
       [2.14349282, 1.90395324, 1.90398769,        nan,        nan],
       [4.25855089, 1.90398488, 1.9039608 ,        nan,        nan],
       [1.53476821, 2.14127377, 2.41948038, 1.90400142,        nan],
       [1.53477342, 1.53477225, 1.

In [ ]:
# # Create the project
# project = Project.from_file(library_path, config_file_name)

# # Run the project
# project.run(
#     which_floris="wind_rose",
#     full_wind_rose=False,  # Only use the O&M analysis period wind data
#     cut_in_wind_speed=3.0,
#     cut_out_wind_speed=25.0,
# )

# # Show the results without considering availability
# print(f"Unadjusted AEP: {project.aep_mwh / 1000:,.2f} GWh")

ORBIT library intialized at '/Users/rhammond/Documents/GitHub/WHaLE/examples/library'
